In [ ]:
# EURUSD Fair-Value and Liquidity Research
## Notebook 00 — Environment and data audit

In [ ]:
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError("This project requires Python 3.11 or newer.")

package_names = (
    "jupyterlab",
    "notebook",
    "ipykernel",
    "numpy",
    "pandas",
    "pyarrow",
    "duckdb",
    "matplotlib",
    "pytest",
    "tzdata",
    "scikit-learn",
    "pip",
    "setuptools",
)

installed_packages = {}
for name in package_names:
    try:
        installed_packages[name] = version(name)
    except PackageNotFoundError:
        installed_packages[name] = None

environment = {
    "python_version": platform.python_version(),
    "python_implementation": platform.python_implementation(),
    "python_executable": sys.executable,
    "operating_system": f"{platform.system()} {platform.release()}",
    "working_directory": str(Path.cwd()),
    "installed_packages": installed_packages,
    "third_party_imports_tested": False,
}

print("ENVIRONMENT")
for key, value in environment.items():
    if key != "installed_packages":
        print(f"{key}: {value}")

print("\nPACKAGE METADATA")
for name, value in installed_packages.items():
    print(f"{name:<16} {value or 'NOT INSTALLED'}")

In [ ]:
## Repository and completed inputs

In [ ]:
import json
import shutil
import subprocess
from itertools import islice

working_directory = Path.cwd().resolve()
candidates = (working_directory, *working_directory.parents)

REPO_ROOT = next(
    (path for path in candidates if (path / ".git").exists()),
    None,
)

if REPO_ROOT is None:
    entered_path = input("Enter the existing repository's full path: ").strip()
    REPO_ROOT = Path(entered_path).expanduser().resolve()

if not (REPO_ROOT / ".git").exists():
    raise RuntimeError("The selected directory has no existing .git marker.")

local_config_path = REPO_ROOT / "configs" / "research_config.local.json"
existing_local = {}

if local_config_path.is_file():
    existing_local = json.loads(
        local_config_path.read_text(encoding="utf-8-sig")
    )
    if not isinstance(existing_local, dict):
        raise ValueError("Local configuration must be a JSON object.")

configured_raw_path = Path(
    existing_local.get("raw_data_dir", "data")
).expanduser()
RAW_DIR = (
    configured_raw_path
    if configured_raw_path.is_absolute()
    else REPO_ROOT / configured_raw_path
).resolve()

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Raw-data directory not found: {RAW_DIR}")

COMPLETED_FILES = [
    "EURUSD_Ticks_2016.01.04_2017.01.02.csv",
    "EURUSD_Ticks_2019.01.02_2021.01.02.csv",
    "EURUSD_Ticks_2021.01.02_2023.01.02.csv",
    "EURUSD_Ticks_2023.01.02_2025.01.02.csv",
    "EURUSD_Ticks_2025.01.02_2026.10.17.csv",
]

SAMPLE_ROWS = 10_000
MAX_LINE_BYTES = 4_096
MAX_SAMPLE_BYTES = 2_000_000
AUDIT_GAP_SECONDS = 30
PIP_SIZE = 0.0001

selected_paths = [RAW_DIR / name for name in COMPLETED_FILES]
missing_files = [path.name for path in selected_paths if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Selected completed files are missing: {missing_files}")

print(f"Repository: {REPO_ROOT}")
print(f"Raw data:   {RAW_DIR}")

print("\nEXISTING REPOSITORY CONTENTS — first 30 entries")
for entry in islice(REPO_ROOT.iterdir(), 30):
    print(f"{'[directory]' if entry.is_dir() else '[file]':<12} {entry.name}")

print("\nRAW CSV INVENTORY — first 40 filenames")
for path in islice(sorted(RAW_DIR.glob("*.csv")), 40):
    selected = path.name in COMPLETED_FILES
    print(
        f"{'SELECTED' if selected else 'NOT SELECTED':<13} "
        f"{path.name} | {path.stat().st_size:,} bytes"
    )

git_executable = shutil.which("git")
tracked_raw_files = []

if git_executable:
    root_check = subprocess.run(
        [git_executable, "-C", str(REPO_ROOT), "rev-parse", "--show-toplevel"],
        capture_output=True,
        text=True,
        check=True,
        timeout=15,
    )
    if Path(root_check.stdout.strip()).resolve() != REPO_ROOT:
        raise RuntimeError("Git reports a different repository root.")

    relative_raw_paths = [
        path.relative_to(REPO_ROOT).as_posix()
        for path in selected_paths
        if path.is_relative_to(REPO_ROOT)
    ]
    if relative_raw_paths:
        tracked_check = subprocess.run(
            [git_executable, "-C", str(REPO_ROOT), "ls-files", "-z",
             "--", *relative_raw_paths],
            capture_output=True,
            text=True,
            check=True,
            timeout=15,
        )
        tracked_raw_files = list(filter(None, tracked_check.stdout.split("\0")))
else:
    print("\nGit CLI is unavailable; the .git marker exists, but CLI verification is pending.")

print(f"\nSelected raw paths currently indexed by Git: {tracked_raw_files}")

In [ ]:
import csv
import io

def read_prefix(path):
    before = path.stat()

    with path.open("rb") as handle:
        header_bytes = handle.readline(MAX_LINE_BYTES + 1)
        if not header_bytes or len(header_bytes) > MAX_LINE_BYTES:
            raise ValueError(f"Missing or oversized header: {path.name}")

        body_lines = []
        body_bytes = 0
        stop_reason = "record_limit"

        for _ in range(SAMPLE_ROWS):
            line = handle.readline(MAX_LINE_BYTES + 1)
            if not line:
                stop_reason = "end_of_file"
                break
            if len(line) > MAX_LINE_BYTES:
                raise ValueError(f"Oversized physical line in {path.name}")
            if body_bytes + len(line) > MAX_SAMPLE_BYTES:
                stop_reason = "byte_limit"
                break
            body_lines.append(line)
            body_bytes += len(line)

        if stop_reason == "record_limit" and not handle.read(1):
            stop_reason = "end_of_file"

    after = path.stat()
    if (before.st_size, before.st_mtime_ns) != (
        after.st_size, after.st_mtime_ns
    ):
        raise RuntimeError(
            f"{path.name} changed during inspection. "
            "Exclude it until the export has finished."
        )

    text = (header_bytes + b"".join(body_lines)).decode("utf-8-sig")
    detection_sample = "\n".join(text.splitlines()[:20])
    dialect = csv.Sniffer().sniff(detection_sample, delimiters=",;\t|")
    reader = csv.reader(io.StringIO(text, newline=""), dialect, strict=True)
    header = next(reader)
    rows = list(reader)

    if not rows:
        raise ValueError(f"No complete data records sampled from {path.name}")

    return {
        "file": path.name,
        "bytes": before.st_size,
        "mtime_ns": before.st_mtime_ns,
        "header": header,
        "delimiter": dialect.delimiter,
        "rows": rows,
        "sample_body_bytes": body_bytes,
        "header_bytes": len(header_bytes),
        "stop_reason": stop_reason,
        "stable_during_read": True,
    }

In [ ]:
import math
from collections import Counter
from datetime import datetime
from statistics import median

EXPECTED_HEADER = ["Time (EET)", "Ask", "Bid", "AskVolume", "BidVolume"]
TIMESTAMP_FORMAT = "%Y.%m.%d %H:%M:%S.%f"

def profile_sample(sample):
    if sample["header"] != EXPECTED_HEADER:
        raise ValueError(
            f"Schema differs in {sample['file']}: {sample['header']}. "
            "Return this error before continuing."
        )

    rows = sample["rows"]
    numeric_names = EXPECTED_HEADER[1:]
    numeric_checks = {
        name: dict(blank=0, invalid=0, nonfinite=0, zero=0, negative=0)
        for name in numeric_names
    }
    flags = dict(
        wrong_field_count=0,
        missing_timestamps=0,
        unparseable_timestamps=0,
        invalid_price_rows=0,
        crossed_quotes=0,
        locked_quotes=0,
        missing_or_invalid_size_rows=0,
        zero_size_rows=0,
        negative_size_rows=0,
        zero_total_size_rows=0,
    )
    times, spreads = [], []
    fraction_digits = Counter()

    for row in rows:
        if len(row) != len(EXPECTED_HEADER):
            flags["wrong_field_count"] += 1
            continue

        timestamp_text = row[0].strip()
        if not timestamp_text:
            flags["missing_timestamps"] += 1
        else:
            try:
                times.append(datetime.strptime(timestamp_text, TIMESTAMP_FORMAT))
                fraction_digits[len(timestamp_text.rsplit(".", 1)[-1])] += 1
            except ValueError:
                flags["unparseable_timestamps"] += 1

        numbers = {}
        for name, text in zip(numeric_names, row[1:]):
            text = text.strip()
            value = None
            if not text:
                numeric_checks[name]["blank"] += 1
            else:
                try:
                    parsed = float(text)
                except ValueError:
                    numeric_checks[name]["invalid"] += 1
                else:
                    if not math.isfinite(parsed):
                        numeric_checks[name]["nonfinite"] += 1
                    else:
                        value = parsed
                        numeric_checks[name]["zero"] += int(value == 0)
                        numeric_checks[name]["negative"] += int(value < 0)
            numbers[name] = value

        ask, bid = numbers["Ask"], numbers["Bid"]
        if ask is None or bid is None or ask <= 0 or bid <= 0:
            flags["invalid_price_rows"] += 1
        else:
            flags["crossed_quotes"] += int(bid > ask)
            flags["locked_quotes"] += int(bid == ask)
            spreads.append((ask - bid) / PIP_SIZE)

        sizes = [numbers["AskVolume"], numbers["BidVolume"]]
        flags["missing_or_invalid_size_rows"] += int(
            any(value is None for value in sizes)
        )
        flags["zero_size_rows"] += int(any(value == 0 for value in sizes))
        flags["negative_size_rows"] += int(
            any(value is not None and value < 0 for value in sizes)
        )
        flags["zero_total_size_rows"] += int(
            all(value is not None and value >= 0 for value in sizes)
            and sum(sizes) == 0
        )

    gaps = [(right - left).total_seconds() for left, right in zip(times, times[1:])]
    flags["backward_source_clock_steps"] = sum(value < 0 for value in gaps)
    flags["gaps_above_audit_threshold"] = sum(
        value > AUDIT_GAP_SECONDS for value in gaps
    )
    flags["repeated_timestamp_rows"] = len(times) - len(set(times))
    flags["duplicate_field_rows"] = len(rows) - len({tuple(row) for row in rows})

    average_record_bytes = sample["sample_body_bytes"] / len(rows)
    estimated_rows = round(
        (sample["bytes"] - sample["header_bytes"]) / average_record_bytes
    )

    return {
        key: value for key, value in sample.items()
        if key not in {"rows", "sample_body_bytes", "header_bytes"}
    } | {
        "sample_rows": len(rows),
        "estimated_file_rows_from_prefix": estimated_rows,
        "sample_source_time_min": min(times).isoformat() if times else None,
        "sample_source_time_max": max(times).isoformat() if times else None,
        "timestamp_fraction_digits": dict(fraction_digits),
        "maximum_forward_gap_seconds": max(
            (value for value in gaps if value >= 0), default=None
        ),
        "flags": flags,
        "numeric_checks": numeric_checks,
        "spread_pips": {
            "minimum": round(min(spreads), 6) if spreads else None,
            "median": round(median(spreads), 6) if spreads else None,
            "maximum": round(max(spreads), 6) if spreads else None,
        },
        "first_records": rows[:3],
    }

In [ ]:
from time import perf_counter

audit_started = perf_counter()
file_audits = []

for path in selected_paths:
    result = profile_sample(read_prefix(path))
    file_audits.append(result)

    print(f"\nFILE: {result['file']}")
    print(f"Header: {result['header']}")
    print(f"Delimiter: {result['delimiter']!r}")
    print(f"Sampled records: {result['sample_rows']:,}")
    print(f"Sampling stopped because: {result['stop_reason']}")
    print(
        "Approximate file records from prefix width: "
        f"{result['estimated_file_rows_from_prefix']:,}"
    )
    print(
        "Sample source-clock coverage: "
        f"{result['sample_source_time_min']} → "
        f"{result['sample_source_time_max']}"
    )
    print("Flags:", result["flags"])
    print("Spread pips:", result["spread_pips"])
    print("First record:", result["first_records"][0])

audit_elapsed_seconds = perf_counter() - audit_started
print(f"\nBounded sample audit runtime: {audit_elapsed_seconds:.3f} seconds")

In [ ]:
data_dictionary = [
    {
        "source_field": "Time (EET)",
        "meaning": "Timestamp shared by the supplied quote snapshot",
        "units": "Formatted source-clock date/time; fractional precision inspected",
        "status": "UTC conversion blocked pending export timezone/DST verification",
    },
    {
        "source_field": "Ask",
        "meaning": "Supplied ask price",
        "units": "USD per EUR under the EURUSD quotation convention",
        "status": "No price rescaling applied",
    },
    {
        "source_field": "Bid",
        "meaning": "Supplied bid price",
        "units": "USD per EUR under the EURUSD quotation convention",
        "status": "No price rescaling applied",
    },
    {
        "source_field": "AskVolume",
        "meaning": "Candidate displayed size associated with the supplied ask",
        "units": "Unresolved export denomination",
        "status": "Not interpreted as executed trading volume",
    },
    {
        "source_field": "BidVolume",
        "meaning": "Candidate displayed size associated with the supplied bid",
        "units": "Unresolved export denomination",
        "status": "Not interpreted as executed trading volume",
    },
]

unresolved_assumptions = [
    "Verify the JForex export timezone and daylight-saving convention.",
    "Verify the CSV displayed-size denomination and any historical changes.",
    "Audit newly completed exports before adding them to ingestion.",
    "Establish full-file coverage, ordering, duplicates, and quality later.",
    "Resolve cross-file overlaps and ordering before modeling.",
    "True side-specific quote ages are unavailable from the observed schema.",
    "Prefix-based file row counts are approximate planning estimates only.",
]

print(json.dumps(data_dictionary, indent=2))
print("\nUNRESOLVED ASSUMPTIONS")
for number, item in enumerate(unresolved_assumptions, start=1):
    print(f"{number}. {item}")

In [ ]:
import hashlib

audit_payload = {
    "notebook": "00_environment_and_data_audit.ipynb",
    "environment": environment,
    "configuration": {
        "sample_rows_limit": SAMPLE_ROWS,
        "max_line_bytes": MAX_LINE_BYTES,
        "max_sample_bytes": MAX_SAMPLE_BYTES,
        "audit_gap_seconds": AUDIT_GAP_SECONDS,
        "pip_size": PIP_SIZE,
        "raw_data_dir": str(RAW_DIR),
        "completed_files": COMPLETED_FILES,
    },
    "files": file_audits,
    "data_dictionary": data_dictionary,
    "unresolved_assumptions": unresolved_assumptions,
    "tracked_raw_files": tracked_raw_files,
    "owner_review": "pending",
    "scope": "bounded prefixes; no full-file ingestion or UTC conversion",
}

audit_text = json.dumps(
    audit_payload, indent=2, sort_keys=True, allow_nan=False
) + "\n"
audit_id = hashlib.sha256(audit_text.encode("utf-8")).hexdigest()[:16]
audit_relative_path = f"artifacts/manifests/notebook00-{audit_id}.json"

example_config = {
    "raw_data_dir": "data",
    "source": {
        "provider": "Dukascopy/JForex",
        "timestamp_column": "Time (EET)",
        "timestamp_format": TIMESTAMP_FORMAT,
        "source_timezone": None,
        "source_timezone_verified": False,
        "displayed_size_units": None,
        "combined_snapshots": True,
    },
    "pip_size": PIP_SIZE,
    "audit_sample_rows": SAMPLE_ROWS,
}
local_config = {"raw_data_dir": str(RAW_DIR)}

quality_lines = [
    "# Preliminary data-quality report",
    "",
    "Scope: bounded prefixes of selected completed exports.",
    "Timestamps remain in the unverified source clock.",
    "No observations were excluded or source files modified.",
    "",
    "| File | Sample records | Duplicate field rows | Crossed | Locked | Median spread, pips |",
    "|---|---:|---:|---:|---:|---:|",
]
for result in file_audits:
    flags = result["flags"]
    quality_lines.append(
        f"| {result['file']} | {result['sample_rows']} | "
        f"{flags['duplicate_field_rows']} | {flags['crossed_quotes']} | "
        f"{flags['locked_quotes']} | {result['spread_pips']['median']} |"
    )

quality_lines += [
    "",
    "These prefixes do not establish representative liquidity conditions, "
    "full coverage, or whole-file quality.",
    "",
    "Quality flags are separate from exclusion decisions. Repeated timestamps "
    "are not automatically duplicate observations. Missing sizes differ from zero sizes.",
    "",
    "## Data dictionary",
]
for field in data_dictionary:
    quality_lines.append(
        f"- **{field['source_field']}**: {field['meaning']}. "
        f"Units: {field['units']}. {field['status']}."
    )
quality_lines += ["", "## Unresolved assumptions"]
quality_lines += [f"- {item}" for item in unresolved_assumptions]
quality_report = "\n".join(quality_lines) + "\n"

In [ ]:
ignore_text = """# Historical and generated data
/data/*
!/data/sample/
/data/sample/*
!/data/sample/synthetic_quotes.csv
/EURUSD_Ticks_*.csv
/artifacts/
/reports/private/
/reports/figures/
/reports/tables/

# Local settings and common credential files
/configs/research_config.local.json
.env
.env.*
*.pem
*.key
credentials/

# Python and notebook caches
__pycache__/
*.py[cod]
*.egg-info/
.pytest_cache/
.ipynb_checkpoints/
.venv/
venv/
build/
dist/
"""

disclosure = (
    "Development approach: This is an AI-assisted research project.\n"
    "Raymond Fendi defined the business objectives and provided FX domain\n"
    "direction. OpenAI Codex assisted with implementation, testing, and\n"
    "documentation. CONTRIBUTIONS.md records completed design decisions,\n"
    "reviews, and validation."
)

readme_sections = {
    "Purpose and business relevance":
        "Investigate short-horizon EURUSD price changes, liquidity deterioration, "
        "and possible electronic-pricing implications.",
    "Data and units":
        "Dukascopy/JForex Level 1 exports. EURUSD pip size is 0.0001. "
        "Source timezone/DST interpretation and displayed-size denomination remain unresolved.",
    "Observed information and limitations":
        "This is a research prototype. Displayed sizes are not executed trade volumes. "
        "A single feed does not represent the entire FX market. Predictive accuracy "
        "does not establish achievable trading profits.",
    "Repository structure":
        "notebooks/: research steps; src/: reusable modules; configs/: settings; "
        "data/: local datasets and safe synthetic samples; tests/: correctness checks; "
        "reports/: findings; artifacts/: generated execution evidence and models.",
    "Setup":
        "Notebook 00 uses Python's standard library. Research dependencies and the "
        "editable package will be established before Notebook 01 imports project code.",
    "Notebook execution order":
        "00 environment/data audit → 01 ingestion/quality → 02 sampling/features → "
        "03 labels/splits → 04 baselines → 05 regimes → 06 walk-forward evaluation → "
        "07 interpretation.",
    "Methodology":
        "Currently implemented: bounded source inspection and prefix-quality diagnostics. "
        "Causal sampling, labels, models, and evaluation remain pending.",
    "Validation":
        "Notebook 00 records execution observations. Third-party imports, research tests, "
        "full-history quality, and owner review remain pending.",
    "Results":
        "Results pending execution and validation.",
    "Development approach":
        disclosure,
    "Next steps":
        "Review Notebook 00 outputs, resolve source assumptions, prepare the package, "
        "and begin synthetic and bounded ingestion checks.",
}
readme_text = "# EURUSD Fair-Value and Liquidity Research\n\n"
readme_text += "\n\n".join(
    f"## {heading}\n\n{body}" for heading, body in readme_sections.items()
) + "\n"

contributions_text = """# Contributions

- Raymond Fendi defined the business objectives and provided FX domain direction.
- OpenAI Codex assisted with the Notebook 00 implementation and documentation.
- Notebook 00 provides bounded environment and source-data inspection.
- Runtime observations are recorded in ignored audit manifests.
- Owner review and understanding checkpoint: pending confirmation.
- Package installation, research tests, ingestion, models, and evaluation: pending.
"""

files_to_create = {
    ".gitignore": ignore_text,
    "configs/research_config.example.json":
        json.dumps(example_config, indent=2, allow_nan=False) + "\n",
    "configs/research_config.local.json":
        json.dumps(local_config, indent=2, allow_nan=False) + "\n",
    "reports/data_quality.md": quality_report,
    "reports/research_results.md":
        "# Research results\n\nResults pending execution and validation.\n",
    "README.md": readme_text,
    "CONTRIBUTIONS.md": contributions_text,
    audit_relative_path: audit_text,
}

# Recheck source metadata before saving evidence.
for result in file_audits:
    current = (RAW_DIR / result["file"]).stat()
    if (current.st_size, current.st_mtime_ns) != (
        result["bytes"], result["mtime_ns"]
    ):
        raise RuntimeError(f"Selected source changed: {result['file']}")

# Preflight every output before writing any output.
for relative_name, text in files_to_create.items():
    path = REPO_ROOT / relative_name
    if not path.resolve().is_relative_to(REPO_ROOT):
        raise RuntimeError(f"Output resolves outside the repository: {relative_name}")
    if path.exists():
        if not path.is_file() or path.read_text(encoding="utf-8-sig") != text:
            raise FileExistsError(
                f"Existing content differs: {relative_name}. "
                "Nothing has been written; return this error before continuing."
            )

for relative_name, text in files_to_create.items():
    path = REPO_ROOT / relative_name
    if path.exists():
        print(f"MATCHES {relative_name}")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(text)
        print(f"CREATED {relative_name}")

In [ ]:
warnings = list(unresolved_assumptions)

if git_executable is None:
    warnings.append("Git CLI verification is pending.")
if tracked_raw_files:
    warnings.append(
        "Selected raw files are indexed by Git; ignore rules do not untrack them."
    )

missing_research_packages = [
    name for name in (
        "numpy", "pandas", "pyarrow", "duckdb",
        "matplotlib", "pytest", "tzdata",
    )
    if installed_packages[name] is None
]
if missing_research_packages:
    warnings.append(
        f"Research packages missing from metadata: {missing_research_packages}"
    )

summary = {
    "notebook": "00_environment_and_data_audit.ipynb",
    "audit_id": audit_id,
    "configuration": {
        "sample_rows_limit_per_file": SAMPLE_ROWS,
        "sample_byte_limit_per_file": MAX_SAMPLE_BYTES,
        "gap_audit_threshold_seconds": AUDIT_GAP_SECONDS,
        "pip_size": PIP_SIZE,
        "timestamp_timezone": "unassigned source clock",
    },
    "python_version": environment["python_version"],
    "installed_packages": installed_packages,
    "input_files_examined": len(file_audits),
    "input_rows_examined": sum(item["sample_rows"] for item in file_audits),
    "processed_dataset_rows_written": 0,
    "sample_runtime_seconds": round(audit_elapsed_seconds, 3),
    "file_summaries": [
        {
            "file": item["file"],
            "bytes": item["bytes"],
            "sample_rows": item["sample_rows"],
            "approximate_file_rows": item["estimated_file_rows_from_prefix"],
            "sample_source_time_min": item["sample_source_time_min"],
            "sample_source_time_max": item["sample_source_time_max"],
            "timestamp_fraction_digits": item["timestamp_fraction_digits"],
            "flags": item["flags"],
            "spread_pips": item["spread_pips"],
        }
        for item in file_audits
    ],
    "validation": {
        "selected_files_read_with_strict_utf8_and_csv_parsing": True,
        "selected_headers_match_observed_schema": True,
        "selected_files_stable_during_checks": True,
        "third_party_imports_tested": False,
        "full_file_coverage_verified": False,
        "owner_review_confirmed": False,
    },
    "warnings": warnings,
    "saved_outputs": list(files_to_create),
}

print("NOTEBOOK_00_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))